# Compare WRs With Respect to Acceleration, Deceleration, and Other Player Information

**<span style="color: gold;">PROJECT</span>**

WR/DB Separation Analysis

**<span style="color: gold;">GOAL OF THIS NOTEBOOK</span>**

In this file we want to compare WRs based on their deceleration ability (from `decel_score`) and acceleration ability (from `accel_score`). We will create a table summarizing these scores and other important player information. These tables will help us identify undervalues, overvalued, and properly valued players with respect to creating separation.

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
# pio.renderers.default = "png"

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from IPython.display import display

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

#### Players Data

In [ ]:
# ======================================
# Load Players Data
# ======================================

players_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "players.csv"), 
        schema_overrides = schema_map["players"]
    )
    # -------------------------------
    # Change Missing draft_overall_pick To "UDFA" 
    # -------------------------------
    .with_columns(
        draft_overall_pick = pl.col("draft_overall_pick").fill_null("UDFA")
    )
    # -------------------------------
    # Only Keep WRs
    # -------------------------------
    .filter(
        pl.col("nfl_position") == "WR"
    )
    # -------------------------------
    # Only Keep Needed Variables
    # -------------------------------
    .select(
        "nfl_id", 
        "display_name", 
        "nfl_position", 
        "draft_round", 
        "draft_overall_pick"
    )
)

print("Players Data:")
print(f"    - Dimensions: {players_df.shape}")
print(f"    - Number of Players: {players_df.select("nfl_id").n_unique()}")
players_df

<br><br/>
#### Combine Features Data

In [ ]:
# =========================================
# Load Combine Data
# =========================================

# -----------------------------------------
# Results Features
# -----------------------------------------

combine_results_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "combine_results.csv"), 
        schema_overrides = schema_map["combine_results"]
    )
    # -------------------------------------
    # Only Keep Needed Variables
    # -------------------------------------
    .select(
        "nfl_id", 
        "ten_yd_split", 
        "forty", 
        "vertical", 
        "broad_jump", 
        "three_cone", 
        "short_shuttle", 
        "ngs_athleticism_score", 
        "ngs_college_production_score", 
        "ngs_final_score"
    )
)

# -----------------------------------------
# Derived Combine Features
# -----------------------------------------

combine_features_df = (
    pl.read_csv(str(SEPARATION_DATA_DIR / "03_WR_Combine_Features_Data.csv"))
    # -------------------------------------
    # Only Keep Needed Variables
    # -------------------------------------
    .select(
        "nfl_id", 
        "accel_score", 
        "decel_score", 
        "cut_score", 
        "master_agility_score"
    )
)

# -----------------------------------------
# Join Combine Data
# -----------------------------------------

combine_df = (
    combine_features_df
    .join(
        combine_results_df, 
        on = "nfl_id", 
        how = "inner"
    )
)

print("Combine Data:")
print(f"    - Dimensions: {combine_df.shape}")
print(f"    - Number of Players: {combine_df.select("nfl_id").n_unique()}")
combine_df

<br><br/>
## Data Cleaning
***

#### Join Player and Combine Data

In [ ]:
# ===================================
# Join Player and Combine Data
# ===================================

analysis_df = (
    players_df
    .join(
        combine_df, 
        on = "nfl_id", 
        how = "left"
    )
)

print("WR and Combine Data:")
print(f"    - Dimensions: {analysis_df.shape}")
print(f"    - Number of Players: {analysis_df.select("nfl_id").n_unique()}")
analysis_df

<br><br/>
## Examine WRs
***

#### Deceleration Score Table

In [ ]:
# ==========================================
# Create Table and Calculate Rank Difference
# ==========================================

display_df = (
    analysis_df
    .select([
        "display_name",
        "decel_score",
        "draft_overall_pick",
        "ngs_athleticism_score",
    ])
    .to_pandas()
)

# Calculate ranks internally (not displayed)
decel_rank = display_df["decel_score"].rank(
    method="min",
    ascending=False,
)

ngs_athleticism_rank = display_df["ngs_athleticism_score"].rank(
    method="min",
    ascending=False,
)

# Positive = deceleration ranks better than NGS athleticism
display_df["Rank Difference"] = (
    ngs_athleticism_rank - decel_rank
)

# Sort by deceleration score
display_df = display_df.sort_values(
    "decel_score",
    ascending=False,
    na_position="last",
)

decel_display_df = display_df


# ==========================================
# Shared Greens Colormap
# ==========================================

greens_cmap = plt.get_cmap("Greens")

def green_color(intensity):
    intensity = max(0.0, min(1.0, intensity))
    return mcolors.to_hex(greens_cmap(intensity))


def text_color_for_intensity(intensity):
    rgb = greens_cmap(max(0.0, min(1.0, intensity)))[:3]

    luminance = (
        0.2126 * rgb[0]
        + 0.7152 * rgb[1]
        + 0.0722 * rgb[2]
    )

    return "black" if luminance > 0.45 else "white"


# ==========================================
# Draft Overall Pick Color Mapping
# ==========================================

def draft_pick_style(value):
    if pd.isna(value):
        return "background-color: #D3D3D3; color: black;"

    value = str(value).strip().upper()

    if value == "UDFA":
        intensity = 0
    else:
        try:
            pick = int(value)
        except (ValueError, TypeError):
            return "background-color: #D3D3D3; color: black;"

        if not 1 <= pick <= 257:
            return "background-color: #D3D3D3; color: black;"

        # Pick 1 = green; pick 257 = white
        intensity = (258 - pick) / 257

    return (
        f"background-color: {green_color(intensity)}; "
        f"color: {text_color_for_intensity(intensity)};"
    )


# ==========================================
# Format and Style Table
# ==========================================

rank_max = display_df["Rank Difference"].abs().max()

if pd.isna(rank_max) or rank_max == 0:
    rank_max = 1

styled_df = (
    display_df.style

    # Number formatting
    .format({
        "decel_score": "{:.2f}",
        "ngs_athleticism_score": "{:.1f}",
        "Rank Difference": "{:+.0f}",
    }, na_rep="—")

    # Deceleration score: 0 = white, 100 = green
    .background_gradient(
        cmap="Greens",
        subset=["decel_score"],
        vmin=0,
        vmax=100,
    )

    # NGS athleticism score: 50 = white, 100 = green
    .background_gradient(
        cmap="Greens",
        subset=["ngs_athleticism_score"],
        vmin=50,
        vmax=100,
    )

    # Draft overall pick: 1 = green, 257 = white
    .map(
        draft_pick_style,
        subset=["draft_overall_pick"],
    )

    # Rank Difference:
    # Green = decel ranks better
    # White = no difference
    # Red = decel ranks worse
    .background_gradient(
        cmap="RdYlGn",
        subset=["Rank Difference"],
        vmin=-rank_max,
        vmax=rank_max,
    )

    # Alignment
    .set_properties(**{
        "text-align": "center",
    })
    .set_properties(
        subset=["display_name"],
        **{"text-align": "left"}
    )

    # Table title
    .set_caption(
        "WR Deceleration Scores vs. NGS Athleticism and Draft Position"
    )
)

display(styled_df)

In [ ]:
# ===============================================
# Save Decel Comparison Table To CSV
# ===============================================

# decel_display_df.pipe(pl.from_pandas).write_csv(str(SEPARATION_DATA_DIR / "31_WR_Decel_Comparison_Table.csv"))

<br><br/>
#### Acceleration Score Table

In [ ]:
# ==========================================
# Create Table and Calculate Rank Difference
# ==========================================

display_df = (
    analysis_df
    .select([
        "display_name",
        "accel_score",
        "draft_overall_pick",
        "ngs_athleticism_score",
    ])
    .to_pandas()
)

# Calculate ranks internally (not displayed)
accel_rank = display_df["accel_score"].rank(
    method="min",
    ascending=False,
)

ngs_athleticism_rank = display_df["ngs_athleticism_score"].rank(
    method="min",
    ascending=False,
)

# Positive = acceleration ranks better than NGS athleticism
display_df["Rank Difference"] = (
    ngs_athleticism_rank - accel_rank
)

# Sort by acceleration score
display_df = display_df.sort_values(
    "accel_score",
    ascending=False,
    na_position="last",
)

accel_display_df = display_df


# ==========================================
# Shared Greens Colormap
# ==========================================

greens_cmap = plt.get_cmap("Greens")

def green_color(intensity):
    intensity = max(0.0, min(1.0, intensity))
    return mcolors.to_hex(greens_cmap(intensity))


def text_color_for_intensity(intensity):
    rgb = greens_cmap(max(0.0, min(1.0, intensity)))[:3]

    luminance = (
        0.2126 * rgb[0]
        + 0.7152 * rgb[1]
        + 0.0722 * rgb[2]
    )

    return "black" if luminance > 0.45 else "white"


# ==========================================
# Draft Overall Pick Color Mapping
# ==========================================

def draft_pick_style(value):
    if pd.isna(value):
        return "background-color: #D3D3D3; color: black;"

    value = str(value).strip().upper()

    if value == "UDFA":
        intensity = 0
    else:
        try:
            pick = int(value)
        except (ValueError, TypeError):
            return "background-color: #D3D3D3; color: black;"

        if not 1 <= pick <= 257:
            return "background-color: #D3D3D3; color: black;"

        # Pick 1 = green; pick 257 = white
        intensity = (258 - pick) / 257

    return (
        f"background-color: {green_color(intensity)}; "
        f"color: {text_color_for_intensity(intensity)};"
    )


# ==========================================
# Format and Style Table
# ==========================================

rank_max = display_df["Rank Difference"].abs().max()

if pd.isna(rank_max) or rank_max == 0:
    rank_max = 1

styled_df = (
    display_df.style

    # Number formatting
    .format({
        "accel_score": "{:.2f}",
        "ngs_athleticism_score": "{:.1f}",
        "Rank Difference": "{:+.0f}",
    }, na_rep="—")

    # Acceleration score: 0 = white, 100 = green
    .background_gradient(
        cmap="Greens",
        subset=["accel_score"],
        vmin=0,
        vmax=100,
    )

    # NGS athleticism score: 50 = white, 100 = green
    .background_gradient(
        cmap="Greens",
        subset=["ngs_athleticism_score"],
        vmin=50,
        vmax=100,
    )

    # Draft overall pick: 1 = green, 257 = white
    .map(
        draft_pick_style,
        subset=["draft_overall_pick"],
    )

    # Rank Difference:
    # Green = acceleration ranks better
    # White = no difference
    # Red = acceleration ranks worse
    .background_gradient(
        cmap="RdYlGn",
        subset=["Rank Difference"],
        vmin=-rank_max,
        vmax=rank_max,
    )

    # Alignment
    .set_properties(**{
        "text-align": "center",
    })
    .set_properties(
        subset=["display_name"],
        **{"text-align": "left"}
    )

    # Table title
    .set_caption(
        "WR Acceleration Scores vs. NGS Athleticism and Draft Position"
    )
)

display(styled_df)

In [ ]:
# ===============================================
# Save Accel Comparison Table To CSV
# ===============================================

# accel_display_df.pipe(pl.from_pandas).write_csv(str(SEPARATION_DATA_DIR / "31_WR_Accel_Comparison_Table.csv"))

<br><br/>
## Deceleration vs Acceleration Among WRs
***

In [ ]:
# ========================================
# Decel Score Data
# ========================================

decel_display_df = (
    decel_display_df
    .pipe(pl.from_pandas)
    .select(
        "display_name", 
        "decel_score"
    )
)

# ========================================
# Accel Score Data
# ========================================

accel_display_df = (
    accel_display_df
    .pipe(pl.from_pandas)
    .select(
        "display_name", 
        "accel_score"
    )
)

In [ ]:
# ===========================================
# Add Separation Data To Metrics
# ===========================================

accel_decel_df = (
    accel_display_df
    .join(
        decel_display_df, 
        on = "display_name", 
        how = "left"
    )
    .join(
        players_df
        .select("display_name", "nfl_id"), 
        on = "display_name", 
        how = "left"
    )
)

accel_decel_df

In [ ]:
# ============================================
# Deceleration vs Accleration In Players
# ============================================

fig = px.scatter(
    data_frame = accel_decel_df, 
    x = "accel_score", 
    y = "decel_score", 
    hover_data = "display_name",
    title = "Acceleration and Deceleration Among WRs In Rookie Cohort", 
    labels = {
        "accel_score": "Acceleration Score", 
        "decel_score": "Deceleration Score"
    },
    template = "simple_white"
)

# Selected players to label
players_to_label = [
    "Ladd McConkey",
    "Rome Odunze",
    "Antwane Wells",
    "Rashee Rice",
    "Tre Tucker",
    "Troy Franklin",
    "Ricky Pearsall",
    "Luke McCaffrey", 
    "Malik Washington", 
    "Keon Coleman"
]

# Keep all points, but remove automatic text labels
fig.update_traces(
    mode="markers",
    marker=dict(size=10)
)

# Manually position labels and draw connector lines
label_offsets = {
    "Ladd McConkey": (25, 25),
    "Rome Odunze": (25, 25),
    "Antwane Wells": (-35, 30),
    "Rashee Rice": (0, -35),
    "Tre Tucker": (35, 15),
    "Troy Franklin": (-35, 15), 
    "Ricky Pearsall": (25, -25),
    "Luke McCaffrey": (-25, 25), 
    "Malik Washington": (25, -25), 
    "Keon Coleman": (60, -10)
}

for player in players_to_label:

    player_data = accel_decel_df.filter(
        pl.col("display_name") == player
    )

    if player_data.height == 0:
        continue

    x = player_data["accel_score"][0]
    y = player_data["decel_score"][0]

    x_offset, y_offset = label_offsets[player]

    fig.add_annotation(
        x=x,
        y=y,
        text=f"<b>{player}</b>",
        showarrow=True,
        arrowhead=0,
        arrowwidth=1,
        arrowcolor="black",
        ax=x_offset,
        ay=-y_offset,
        font=dict(size=11, color="black"),
        # bgcolor="rgba(255,255,255,0.8)",
        borderpad=2
    )



# Get axis ranges
x_min, x_max = accel_decel_df["accel_score"].min(), accel_decel_df["accel_score"].max()
y_min, y_max = accel_decel_df["decel_score"].min(), accel_decel_df["decel_score"].max()

x_mid = (x_min + x_max) / 2
y_mid = (y_min + y_max) / 2

# Bottom-left quadrant: red
fig.add_shape(
    type="rect",
    x0=x_min, x1=x_mid,
    y0=y_min, y1=y_mid,
    fillcolor="red",
    opacity=0.15,
    line_width=0,
    layer="below"
)

# Top-left quadrant: green
fig.add_shape(
    type="rect",
    x0=x_mid, x1=x_max,
    y0=y_mid, y1=y_max,
    fillcolor="green",
    opacity=0.15,
    line_width=0,
    layer="below"
)

# ELITE label (top-right)
fig.add_annotation(
    x=78,
    y=86,
    text="<i><b>ELITE</b></i>",
    showarrow=False,
    font=dict(size=20, color="darkgreen")
)

# SUBPAR label (bottom-left)
fig.add_annotation(
    x=7.5,
    y=6,
    text="<i><b>SUBPAR</b></i>",
    showarrow=False,
    font=dict(size=20, color="darkred")
)

fig.update_layout(
    width=1000,
    height=650
)

fig.show()